### Recurrent Neural Network (LSTM, V2)

This is a basic convolutional approach to sequence prediction using convolutions through TensorFlow!

We begin by important any relevant packages, modules, and the DataProcessor class (for our pre-processed data). Then we get our dataset ready.

In [2]:
import os, sys
sys.path.append(os.path.abspath('..'))  # add parent directory to sys.path
from data_cleanup import DataProcessor
from tensorflow.keras.layers import Dense, LSTM, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

# Define the windowing parameters
# Use 24 hours of history
INPUT_WINDOW = 72
# Predict the next 24 hours
OUTPUT_WINDOW = 24 

# Initialize the class
processor = DataProcessor(input_steps=INPUT_WINDOW, output_steps=OUTPUT_WINDOW)

# Run the pipeline
(X_train, y_train), (X_val, y_val), (X_test, y_test) = processor.load_and_process_data()

# Check the final shapes
print("\n--- Final Data Shapes ---")
print(f"X_train shape: {X_train.shape}  | y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}     | y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}    | y_test shape:  {y_test.shape}")

Step 1/5: Fetching, cleaning, and engineering features...


C:\Users\Jose Romero\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\ucimlrepo\fetch.py:97: DtypeWarning: Columns (2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


Step 2/5: Resampling data to hourly...


c:\Users\Jose Romero\Documents\ECS171G13-master\data_cleanup.py:95: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df.resample('H').agg(agg_dict).fillna(method='ffill')


Step 3/5: Splitting and Scaling (StandardScaler)...
Step 4/5: Creating windows...
Step 5/5: Done.

--- Final Data Shapes ---
X_train shape: (25832, 72, 12)  | y_train shape: (25832, 24)
X_val shape:   (3529, 72, 12)     | y_val shape:   (3529, 24)
X_test shape:  (4943, 72, 12)    | y_test shape:  (4943, 24)


c:\Users\Jose Romero\Documents\ECS171G13-master\data_cleanup.py:95: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_hourly = df.resample('H').agg(agg_dict).fillna(method='ffill')


Next, we want to build the model itself. Keras makes this very simple at a high-level, so tuning is also easy to do.

In [ ]:
from tensorflow.keras.layers import Input, Dense, LSTM, Dropout, RepeatVector, TimeDistributed
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
model = Sequential()

model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))
model.add(LSTM(128, return_sequences=False)) 
model.add(Dropout(0.1))

model.add(RepeatVector(OUTPUT_WINDOW))

model.add(LSTM(128, return_sequences=True))
model.add(Dropout(0.1))

model.add(TimeDistributed(Dense(1)))

model.compile(optimizer=Adam(learning_rate=0.0005), 
              loss='mse', 
              metrics=['mae'])

model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │        72,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 24, 128)        │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 24, 1)          │           129 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 203,905 (796.50 KB)

 Trainable params: 203,905 (796.50 KB)

 Non-trainable params: 0 (0.00 B)

Now we want to fit the model, train it, and determine an error metric.

In [ ]:
es = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50, 
    batch_size=32,
    callbacks=[es],
    verbose=1
)

sample_idx = 50
input_seq = X_test[sample_idx]  # Shape (72, features)
actual_output = y_test[sample_idx] # Shape (24, 1)

pred_scaled = model.predict(input_seq.reshape(1, INPUT_WINDOW, X_test.shape[2]))

pred_unscaled = processor.inverse_transform_predictions(pred_scaled[0].flatten())
actual_unscaled = processor.inverse_transform_predictions(actual_output.flatten())

mse = mean_squared_error(actual_unscaled, pred_unscaled)
rmse = np.sqrt(mse)

print(f'\n--- Model Evaluation ---')
print(f'Test Set RMSE: {rmse:.4f} kW')

Epoch 1/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 19s 23ms/step - loss: 0.5705 - mae: 0.5479 - val_loss: 0.5852 - val_mae: 0.5682
Epoch 2/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - loss: 0.5173 - mae: 0.5175 - val_loss: 0.6021 - val_mae: 0.5727
Epoch 3/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 18s 22ms/step - loss: 0.4734 - mae: 0.4927 - val_loss: 0.6309 - val_mae: 0.5812
Epoch 4/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 18s 22ms/step - loss: 0.4327 - mae: 0.4693 - val_loss: 0.6287 - val_mae: 0.5818
Epoch 5/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 18s 22ms/step - loss: 0.3981 - mae: 0.4491 - val_loss: 0.6536 - val_mae: 0.5875
Epoch 6/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 18s 22ms/step - loss: 0.3686 - mae: 0.4317 - val_loss: 0.7024 - val_mae: 0.6060
Epoch 7/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 17s 22ms/step - loss: 0.3428 - mae: 0.4160 - val_loss: 0.7161 - val_mae: 0.6164
Epoch 8/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 18s 22ms/step - loss: 0.3220 - mae: 0.4035 - val_loss: 0.6998 - val_mae: 0.6079
Epoch 9/50
808/808 ━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
# Graph 1: Loss Curves (MSE)

plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss (MSE)', color='blue')
plt.plot(history.history['val_loss'], label='Validation Loss (MSE)', color='orange')
plt.title('Training and Validation Loss over Epochs (repeat)')
plt.xlabel('Epochs')
plt.ylabel('Loss (Mean Squared Error)')
plt.legend()
plt.grid(True)
plt.show()

# Graph 2: MAE Curves

plt.figure(figsize=(12, 6))
plt.plot(history.history['mae'], label='Training MAE', color='green')
plt.plot(history.history['val_mae'], label='Validation MAE', color='red')
plt.title('Training and Validation Mean Absolute Error (repeat)')
plt.xlabel('Epochs')
plt.ylabel('MAE (Scaled)')
plt.legend()
plt.grid(True)
plt.show()

# Graph 3: Actual vs Predicted (Time Series)

# We take the first 200 hours for a clean zoom-in (like the uploaded image)
subset_n = 200 

# Extract just the 1st hour prediction from each window (index 0)
# This reconstructs a continuous timeline
actual_trace = unscaled_y_test[:subset_n, 0]
pred_trace = unscaled_predictions[:subset_n, 0]

plt.figure(figsize=(14, 6))
plt.plot(actual_trace, label='Actual Power (kW)', color='black', linewidth=2)
plt.plot(pred_trace, label='LSTM Predicted (kW)', color='cyan', linestyle='--')
plt.title(f'Actual vs Predicted Global Active Power (First {subset_n} Test Hours) (repeat)')
plt.xlabel('Time (Hours)')
plt.ylabel('Global Active Power (kW)')
plt.legend()
plt.grid(True)
plt.show()

# Graph 4: 24-Hour Rolling Comparison

import matplotlib.pyplot as plt

sample_idx = 50

# Get the actual 24-hour sequence for this sample
real_24h = unscaled_y_test[sample_idx]

# Get the predicted 24-hour sequence for this sample
pred_24h = unscaled_predictions[sample_idx]

plt.figure(figsize=(10, 5))
plt.plot(real_24h, label='Actual (Real)', marker='o', color='black')
plt.plot(pred_24h, label='Predicted (Repeat LSTM)', marker='x', linestyle='--', color='cyan')

plt.title(f'24-Hour Energy Forecast (Sample #{sample_idx})')
plt.xlabel('Hour of the Day')
plt.ylabel('Global Active Power (kW)')
plt.legend()
plt.grid(True)
plt.show()